In [ ]:
import json
import pickle
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
)

In [ ]:
LOOKBACK = 48
RANDOM_STATE = 42
BATCH_SIZE = 64
EPOCHS = 50

MODEL_NAME = "gru_v1"

DATASET_PATH = "/content/final_training_dataset.csv"

ARTIFACT_DIR = Path("models")
ARTIFACT_DIR.mkdir(exist_ok=True)

In [ ]:
df = pd.read_csv(DATASET_PATH)

print(df.shape)
display(df.head())

(20440, 22)


,Date,Close,High,Low,Open,Volume,RSI,MACD,EMA20,EMA50,...,BB_UPPER,BB_LOWER,BB_WIDTH,ROC,MOMENTUM,DAILY_RETURN,VOLATILITY,VOLUME_CHANGE,Stock,Target
0,2018-03-14,365.454681,368.308448,361.952341,368.308448,4194515,46.119976,-7.927174,365.854612,387.016261,...,379.007279,347.370645,31.636634,-0.693227,-6.831726,-0.008679,0.020436,-0.265766,BHARTIARTL,0
1,2018-03-15,363.595398,366.924767,360.828103,365.454649,3636202,44.884867,-7.215790,365.639449,386.097796,...,377.260732,347.876234,29.384498,-0.638072,-7.004700,-0.005088,0.020445,-0.133105,BHARTIARTL,0
2,2018-03-16,360.395782,369.043552,357.714957,359.098605,8030364,42.762519,-6.831447,365.140052,385.089874,...,376.241562,347.840384,28.401178,-3.193946,-9.512512,-0.008800,0.020568,1.208448,BHARTIARTL,0
3,2018-03-19,345.608032,364.589878,343.056928,361.476671,6000592,34.615841,-7.632122,363.279860,383.541566,...,377.064111,345.283955,31.780156,-6.743675,-18.852173,-0.041032,0.023753,-0.252762,BHARTIARTL,1
4,2018-03-20,346.256622,348.937447,339.597855,345.045945,8280683,35.198954,-8.120717,361.658599,382.079411,...,377.714838,343.426864,34.287974,-6.393929,-11.674530,0.001877,0.023383,0.379978,BHARTIARTL,1


In [ ]:
print("=" * 50)
print("Dataset Information")
print("=" * 50)

print(df.info())

print("\nMissing Values")
print(df.isna().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nUnique Stocks")
print(sorted(df["Stock"].unique()))

print("\nTarget Distribution")
print(df["Target"].value_counts())

Dataset Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20440 entries, 0 to 20439
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           20440 non-null  object 
 1   Close          20440 non-null  float64
 2   High           20440 non-null  float64
 3   Low            20440 non-null  float64
 4   Open           20440 non-null  float64
 5   Volume         20440 non-null  int64  
 6   RSI            20440 non-null  float64
 7   MACD           20440 non-null  float64
 8   EMA20          20440 non-null  float64
 9   EMA50          20440 non-null  float64
 10  ATR            20440 non-null  float64
 11  ADX            20440 non-null  float64
 12  BB_UPPER       20440 non-null  float64
 13  BB_LOWER       20440 non-null  float64
 14  BB_WIDTH       20440 non-null  float64
 15  ROC            20440 non-null  float64
 16  MOMENTUM       20440 non-null  float64
 17  DAILY_RETURN   20440 non-null 

In [ ]:
df = df.replace([np.inf, -np.inf], np.nan)

before_rows = len(df)

df = df.dropna().reset_index(drop=True)

after_rows = len(df)

print(f"Rows before cleaning : {before_rows}")
print(f"Rows after cleaning  : {after_rows}")

Rows before cleaning : 20440
Rows after cleaning  : 20401


In [ ]:
FEATURES = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "RSI",
    "MACD",
    "EMA20",
    "EMA50",
    "ATR",
    "ADX",
    "BB_UPPER",
    "BB_LOWER",
    "BB_WIDTH",
    "ROC",
    "MOMENTUM",
    "DAILY_RETURN",
    "VOLATILITY",
    "VOLUME_CHANGE",
    "Stock",
]

TARGET = "Target"

In [ ]:
stock_encoder = LabelEncoder()

stock_encoder.fit(df["Stock"])

original_stock_names = stock_encoder.classes_.copy()

df["Stock"] = stock_encoder.transform(df["Stock"])

In [ ]:
print("Encoded Classes")

for index, stock in enumerate(stock_encoder.classes_):
    print(index, "->", stock)

Encoded Classes
0 -> BHARTIARTL
1 -> HDFCBANK
2 -> HINDUNILVR
3 -> ICICIBANK
4 -> INFY
5 -> ITC
6 -> LT
7 -> RELIANCE
8 -> SBIN
9 -> TCS


In [ ]:
print(df["Stock"].head())

0    0
1    0
2    0
3    0
4    0
Name: Stock, dtype: int64


In [ ]:
print(df[["Stock"]].head(20))
print(df["Stock"].value_counts().sort_index())

    Stock
0       0
1       0
2       0
3       0
4       0
5       0
6       0
7       0
8       0
9       0
10      0
11      0
12      0
13      0
14      0
15      0
16      0
17      0
18      0
19      0
Stock
0    2040
1    2040
2    2040
3    2040
4    2040
5    2041
6    2040
7    2040
8    2040
9    2040
Name: count, dtype: int64


In [ ]:
X_features = df[FEATURES].copy()
y_target = df[TARGET].copy()

In [ ]:
def create_sequences(
    dataframe,
    feature_columns,
    target_column,
    lookback,
):
    """
    Create sequential samples for GRU training.

    Returns
    -------
    X
        Feature sequences

    y
        Targets

    sequence_stock
        Original stock name for each sequence

    sequence_date
        Last date of each sequence

    sequence_df_index
        Original dataframe index
    """

    X = []
    y = []

    sequence_stock = []
    sequence_date = []
    sequence_df_index = []

    grouped = dataframe.groupby("Stock", sort=False)

    for stock_id, stock_df in grouped:

        stock_df = stock_df.reset_index(drop=False)

        feature_values = stock_df[feature_columns].values
        target_values = stock_df[target_column].values

        for i in range(lookback, len(stock_df)):

            X.append(
                feature_values[
                    i - lookback : i
                ]
            )

            y.append(
                target_values[i]
            )

            sequence_stock.append(stock_id)

            sequence_date.append(
                stock_df.iloc[i]["Date"]
            )

            sequence_df_index.append(
                stock_df.iloc[i]["index"]
            )

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.int64),
        np.asarray(sequence_stock),
        np.asarray(sequence_date),
        np.asarray(sequence_df_index),
    )

In [ ]:
(
    X,
    y,
    sequence_stock,
    sequence_date,
    sequence_df_index,
) = create_sequences(
    dataframe=df,
    feature_columns=FEATURES,
    target_column=TARGET,
    lookback=LOOKBACK,
)

In [ ]:
print("=" * 60)

print("Sequence Verification")

print("=" * 60)

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")

print()

print(f"Sequence length : {LOOKBACK}")

print(f"Feature count : {X.shape[2]}")

print()

print("Target Distribution")

print(pd.Series(y).value_counts())

print()

print("Unique Encoded Stocks")

print(np.unique(sequence_stock))

Sequence Verification
X shape : (19921, 48, 20)
y shape : (19921,)

Sequence length : 48
Feature count : 20

Target Distribution
1    10135
0     9786
Name: count, dtype: int64

Unique Encoded Stocks
[0 1 2 3 4 5 6 7 8 9]


In [ ]:
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15

assert abs(
    TRAIN_RATIO +
    VALIDATION_RATIO +
    TEST_RATIO -
    1.0
) < 1e-9

In [ ]:
total_sequences = len(X)

train_end = int(
    total_sequences * TRAIN_RATIO
)

validation_end = (
    train_end +
    int(
        total_sequences * VALIDATION_RATIO
    )
)

In [ ]:
X_train = X[:train_end]
y_train = y[:train_end]

X_validation = X[
    train_end:validation_end
]
y_validation = y[
    train_end:validation_end
]

X_test = X[
    validation_end:
]
y_test = y[
    validation_end:
]

In [ ]:
stock_train = sequence_stock[:train_end]
stock_validation = sequence_stock[
    train_end:validation_end
]
stock_test = sequence_stock[
    validation_end:
]

date_train = sequence_date[:train_end]
date_validation = sequence_date[
    train_end:validation_end
]
date_test = sequence_date[
    validation_end:
]

In [ ]:
print("=" * 60)
print("Temporal Split Verification")
print("=" * 60)

print()

print(
    f"Training   : {len(X_train)}"
)

print(
    f"Validation : {len(X_validation)}"
)

print(
    f"Test       : {len(X_test)}"
)

print()

print(
    f"Training Shape   : {X_train.shape}"
)

print(
    f"Validation Shape : {X_validation.shape}"
)

print(
    f"Test Shape       : {X_test.shape}"
)

print()

print(
    f"Total Samples : "
    f"{len(X_train)+len(X_validation)+len(X_test)}"
)

Temporal Split Verification

Training   : 13944
Validation : 2988
Test       : 2989

Training Shape   : (13944, 48, 20)
Validation Shape : (2988, 48, 20)
Test Shape       : (2989, 48, 20)

Total Samples : 19921


In [ ]:
def scale_sequences(
    scaler,
    sequence_data,
    fit=False,
):
    """
    Scale 3D sequence data while preserving shape.

    Parameters
    ----------
    scaler : MinMaxScaler

    sequence_data : ndarray
        Shape:
        (samples, lookback, features)

    fit : bool
        True  -> fit + transform
        False -> transform only

    Returns
    -------
    scaled_data
        Same shape as input.
    """

    original_shape = sequence_data.shape

    flattened = sequence_data.reshape(
        -1,
        original_shape[-1]
    )

    if fit:
        flattened = scaler.fit_transform(flattened)
    else:
        flattened = scaler.transform(flattened)

    scaled_data = flattened.reshape(
        original_shape
    )

    return scaled_data

In [ ]:
feature_scaler = MinMaxScaler()

In [ ]:
X_train = scale_sequences(
    scaler=feature_scaler,
    sequence_data=X_train,
    fit=True,
)

In [ ]:
X_validation = scale_sequences(
    scaler=feature_scaler,
    sequence_data=X_validation,
    fit=False,
)

In [ ]:
X_test = scale_sequences(
    scaler=feature_scaler,
    sequence_data=X_test,
    fit=False,
)

In [ ]:
print("=" * 60)
print("Scaling Verification")
print("=" * 60)

print()

print("Training Shape")
print(X_train.shape)

print()

print("Validation Shape")
print(X_validation.shape)

print()

print("Test Shape")
print(X_test.shape)

print()

print("Training Min :", X_train.min())
print("Training Max :", X_train.max())

print()

print("Validation Min :", X_validation.min())
print("Validation Max :", X_validation.max())

print()

print("Test Min :", X_test.min())
print("Test Max :", X_test.max())

Scaling Verification

Training Shape
(13944, 48, 20)

Validation Shape
(2988, 48, 20)

Test Shape
(2989, 48, 20)

Training Min : 0.0
Training Max : 1.0000001

Validation Min : 0.0
Validation Max : 1.3333334

Test Min : 0.0
Test Max : 1.5


In [ ]:
def build_gru_model(
    input_shape,
):
    """
    Build the production GRU model.
    """

    model = Sequential(

        [

            GRU(
                128,
                return_sequences=True,
                input_shape=input_shape,
            ),

            Dropout(
                0.3
            ),

            GRU(
                64,
            ),

            Dropout(
                0.3
            ),

            Dense(
                32,
                activation="relu",
            ),

            Dense(
                1,
                activation="sigmoid",
            ),

        ]
    )

    return model

In [ ]:
model = build_gru_model(
    input_shape=(
        LOOKBACK,
        len(FEATURES),
    )
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 48, 128)        │        57,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 48, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 96,961 (378.75 KB)

 Trainable params: 96,961 (378.75 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
    ],
)

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1,
)

model_checkpoint = ModelCheckpoint(
    filepath=ARTIFACT_DIR / "gru_v1.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1,
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(
        X_validation,
        y_validation,
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        early_stopping,
        model_checkpoint,
    ],
    verbose=1,
)

Epoch 1/50
216/218 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5031 - loss: 0.6951
Epoch 1: val_loss improved from None to 0.69264, saving model to models/gru_v1.keras

Epoch 1: finished saving model to models/gru_v1.keras
218/218 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - accuracy: 0.5018 - loss: 0.6949 - val_accuracy: 0.5171 - val_loss: 0.6926
Epoch 2/50
216/218 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4998 - loss: 0.6939
Epoch 2: val_loss improved from 0.69264 to 0.69261, saving model to models/gru_v1.keras

Epoch 2: finished saving model to models/gru_v1.keras
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.4937 - loss: 0.6943 - val_accuracy: 0.5211 - val_loss: 0.6926
Epoch 3/50
215/218 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4978 - loss: 0.6935
Epoch 3: val_loss did not improve from 0.69261
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.5001 - loss: 0.6936 - val_accuracy: 0.5097 - val_loss: 0.6931
Epoch 4/50
216/218 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - acc

In [ ]:
print("=" * 60)
print("Training Complete")
print("=" * 60)

print()

print(
    "Best Validation Accuracy:",
    max(history.history["val_accuracy"])
)

print()

print(
    "Best Validation Loss:",
    min(history.history["val_loss"])
)

Training Complete

Best Validation Accuracy: 0.5210843086242676

Best Validation Loss: 0.6926075220108032


In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=1,
)

print("=" * 60)
print("Test Results")
print("=" * 60)

print(f"Test Loss     : {test_loss:.6f}")
print(f"Test Accuracy : {test_accuracy:.6f}")

94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5032 - loss: 0.6927
Test Results
Test Loss     : 0.692652
Test Accuracy : 0.503178


In [ ]:
y_probability = model.predict(
    X_test,
    verbose=1,
)

y_prediction = (
    y_probability >= 0.5
).astype(int)

94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step


In [ ]:
print("=" * 60)
print("Classification Report")
print("=" * 60)

print(
    classification_report(
        y_test,
        y_prediction,
        digits=4,
    )
)

Classification Report
              precision    recall  f1-score   support

           0     0.4914    0.6256    0.5504      1453
           1     0.5224    0.3874    0.4449      1536

    accuracy                         0.5032      2989
   macro avg     0.5069    0.5065    0.4976      2989
weighted avg     0.5073    0.5032    0.4962      2989



In [ ]:
confusion = confusion_matrix(
    y_test,
    y_prediction,
)

print("=" * 60)
print("Confusion Matrix")
print("=" * 60)

print(confusion)

Confusion Matrix
[[909 544]
 [941 595]]


In [ ]:
final_accuracy = accuracy_score(
    y_test,
    y_prediction,
)

print("=" * 60)
print("Final Accuracy")
print("=" * 60)

print(f"{final_accuracy:.6f}")

Final Accuracy
0.503178


In [ ]:
confusion = confusion_matrix(
    y_test,
    y_prediction,
)

print("=" * 60)
print("Confusion Matrix")
print("=" * 60)

print(confusion)

Confusion Matrix
[[909 544]
 [941 595]]


In [ ]:
final_accuracy = accuracy_score(
    y_test,
    y_prediction,
)

print("=" * 60)
print("Final Accuracy")
print("=" * 60)

print(f"{final_accuracy:.6f}")

Final Accuracy
0.503178


In [ ]:
model.save(
    ARTIFACT_DIR / "gru_v1.keras"
)

In [ ]:
joblib.dump(
    feature_scaler,
    ARTIFACT_DIR / "feature_scaler.pkl",
)

['models/feature_scaler.pkl']

In [ ]:
joblib.dump(
    stock_encoder,
    ARTIFACT_DIR / "stock_encoder.pkl",
)

['models/stock_encoder.pkl']

In [ ]:
with open(
    ARTIFACT_DIR / "history.pkl",
    "wb",
) as file:

    pickle.dump(
        history.history,
        file,
    )

In [ ]:
metadata = {

    "model_name": MODEL_NAME,

    "sequence_length": LOOKBACK,

    "feature_count": len(FEATURES),

    "features": FEATURES,

    "target_column": TARGET,

    "stock_classes": stock_encoder.classes_.tolist(),

    "train_samples": len(X_train),

    "validation_samples": len(X_validation),

    "test_samples": len(X_test),

    "training_accuracy": float(
        max(
            history.history["accuracy"]
        )
    ),

    "validation_accuracy": float(
        max(
            history.history["val_accuracy"]
        )
    ),

    "test_accuracy": float(
        final_accuracy
    ),

}

In [ ]:
with open(
    ARTIFACT_DIR / "model_metadata.json",
    "w",
) as file:

    json.dump(
        metadata,
        file,
        indent=4,
    )

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    history.history["accuracy"],
    label="Training",
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation",
)

plt.title("Accuracy")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend()

plt.grid(True)

plt.tight_layout()

plt.savefig(
    ARTIFACT_DIR / "accuracy_curve.png"
)

plt.close()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    history.history["loss"],
    label="Training",
)

plt.plot(
    history.history["val_loss"],
    label="Validation",
)

plt.title("Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend()

plt.grid(True)

plt.tight_layout()

plt.savefig(
    ARTIFACT_DIR / "loss_curve.png"
)

plt.close()

In [ ]:
print("=" * 60)
print("Artifact Verification")
print("=" * 60)

artifacts = [
    "gru_v1.keras",
    "feature_scaler.pkl",
    "stock_encoder.pkl",
    "model_metadata.json",
    "history.pkl",
    "accuracy_curve.png",
    "loss_curve.png",
]

for artifact in artifacts:

    path = ARTIFACT_DIR / artifact

    print(
        f"{artifact:<25} :",
        "FOUND" if path.exists() else "MISSING",
    )

print()

print("Encoder Classes")

print(stock_encoder.classes_)

Artifact Verification
gru_v1.keras              : FOUND
feature_scaler.pkl        : FOUND
stock_encoder.pkl         : FOUND
model_metadata.json       : FOUND
history.pkl               : FOUND
accuracy_curve.png        : FOUND
loss_curve.png            : FOUND

Encoder Classes
['BHARTIARTL' 'HDFCBANK' 'HINDUNILVR' 'ICICIBANK' 'INFY' 'ITC' 'LT'
 'RELIANCE' 'SBIN' 'TCS']
